## Root Cause Analysis - Exp4

* Goals
  * Check whether can provide improvement suggestions at individual cases level


In [1]:
%load_ext autoreload
%autoreload 2

import os
import pandas as pd
import psycopg2
from utils_root_cause import upload_to_google_drive

import warnings
warnings.filterwarnings('ignore')
warnings.filterwarnings("ignore", message="None of PyTorch, TensorFlow")

None of PyTorch, TensorFlow >= 2.0, or Flax have been found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


In [ ]:
# Environment variable holding the database URL
DB_ENV = "DATABASE_URL_PUBLIC_RAG_DR"
db_url = os.getenv(DB_ENV)
if not db_url:
    raise RuntimeError(f"Environment variable {DB_ENV} is not set")

# Connect and load the table into a pandas DataFrame
conn = psycopg2.connect(db_url)
try:
    df_existing_rca_output = pd.read_sql_query("SELECT * FROM existing_rca_output", conn)
finally:
    try:
        conn.close()
    except Exception:
        pass

# Display the first rows
print(df_existing_rca_output.shape)
df_existing_rca_output.head()

(3, 5)


,id,config_hash,rca_records,agg_review,created_at
0,1,fac62bbce5555de5ca44b01442baea3f,[{'query': 'How to deposit a cheque issued to ...,{'root_cause_analysis': 'LLM omits critical de...,2026-04-16 00:47:56.728249
1,2,93f67870e1be18f0328798b37faf07df,[{'query': 'How to deposit a cheque issued to ...,{'root_cause_analysis': 'LLM omits critical de...,2026-04-16 00:49:09.847485
2,3,cda0e838de11a82a1802e38b8b0d9ab9,[{'query': 'How to deposit a cheque issued to ...,{'root_cause_analysis': 'Low semantic weight &...,2026-04-17 02:40:40.743896


In [14]:
# Convert any list-valued columns so each item becomes its own row
df = df_existing_rca_output.copy()
# Find columns that contain lists in at least one row
list_cols = [c for c in df.columns if df[c].apply(lambda x: isinstance(x, list)).any()]
if not list_cols:
    print('No list-valued columns found; no conversion needed')
else:
    for col in list_cols:
        # Explode the list so each element becomes a separate row
        df = df.explode(col).reset_index(drop=True)
        # If the exploded elements are dict-like, expand their keys into columns
        if df[col].dropna().apply(lambda x: isinstance(x, dict)).any():
            expanded = pd.json_normalize(df[col]).add_prefix(f'{col}.')
            df = pd.concat([df.drop(columns=[col]).reset_index(drop=True), expanded.reset_index(drop=True)], axis=1)
    df_records_rows = df
    print('Converted list-valued columns to rows. Result shape:', df_records_rows.shape)
    display(df_records_rows.head())

Converted list-valued columns to rows. Result shape: (90, 18)


,id,config_hash,agg_review,created_at,rca_records.query,rca_records.context,rca_records.ai_answer,rca_records.aq_reasoning,rca_records.rq_reasoning,rca_records.same_context,rca_records.needs_re_eval,rca_records.query_quality,rca_records.referenced_answer,rca_records.retrieved_content,rca_records.root_cause_analysis,rca_records.retrieval_quality_score,rca_records.new_answer_quality_score,rca_records.new_retrieval_quality_score
0,1,fac62bbce5555de5ca44b01442baea3f,{'root_cause_analysis': 'LLM omits critical de...,2026-04-16 00:47:56.728249,How to deposit a cheque issued to an associate...,Just have the associate sign the back and then...,To deposit a cheque issued to an associate in ...,The AI’s answer correctly advises obtaining th...,The retrieved content is essentially identical...,True,0,clear,Have the check reissued to the proper payee.Ju...,Just have the associate sign the back and then...,[],3,2,3
1,1,fac62bbce5555de5ca44b01442baea3f,{'root_cause_analysis': 'LLM omits critical de...,2026-04-16 00:47:56.728249,Can I send a money order from USPS as a business?,Sure you can. You can fill in whatever you wa...,"Yes, you can fill in your business name and ad...",The AI’s answer correctly confirms that a busi...,The retrieved content includes all the critica...,True,0,clear,Sure you can. You can fill in whatever you wa...,Sure you can. You can fill in whatever you wa...,[],3,2,3
2,1,fac62bbce5555de5ca44b01442baea3f,{'root_cause_analysis': 'LLM omits critical de...,2026-04-16 00:47:56.728249,1 EIN doing business under multiple business n...,You're confusing a lot of things here. Company...,"In some cases, a single EIN can be used for a ...","The referenced answer discusses merging LLCs, ...",The retrieved content is identical to the cont...,True,1,clear,You're confusing a lot of things here. Company...,You're confusing a lot of things here. Company...,[{'Please review referenced answer': 'referenc...,3,-1,3
3,1,fac62bbce5555de5ca44b01442baea3f,{'root_cause_analysis': 'LLM omits critical de...,2026-04-16 00:47:56.728249,Applying for and receiving business credit,Set up a meeting with the bank that handles yo...,Establishing a relationship with a banker is c...,The AI’s answer addresses the user’s question ...,The retrieved content is a near‑identical copy...,True,0,clear,"""I'm afraid the great myth of limited liabilit...",Set up a meeting with the bank that handles yo...,[],3,2,3
4,1,fac62bbce5555de5ca44b01442baea3f,{'root_cause_analysis': 'LLM omits critical de...,2026-04-16 00:47:56.728249,401k Transfer After Business Closure,The time horizon for your 401K/IRA is essentia...,If your employer's 401(k) plan is terminated d...,The AI’s answer is relevant to the query—it su...,The retrieved content is essentially identical...,False,0,clear,You should probably consult an attorney. Howev...,The time horizon for your 401K/IRA is essentia...,[],3,2,3


In [15]:
folder_id = os.getenv('GOOGLE_DRIVE_FOLDER_ID')
upload_to_google_drive(df_records_rows, folder_id, 'rca_revisit_v1.xlsx')

Overwritten file ID: 1PQyr8-ocML3TKCChTisH-BqYh-H6VHQp


## Data Analysis

In [16]:
dist_df = pd.DataFrame(df_records_rows[['config_hash', 
                'rca_records.new_retrieval_quality_score',
                'rca_records.new_answer_quality_score']].value_counts()).reset_index()
dist_df.sort_values(by=['config_hash', 'count'], ascending=[True, False], inplace=True)

dist_df

,config_hash,rca_records.new_retrieval_quality_score,rca_records.new_answer_quality_score,count
0,93f67870e1be18f0328798b37faf07df,3,3,18
3,93f67870e1be18f0328798b37faf07df,3,2,9
12,93f67870e1be18f0328798b37faf07df,3,1,1
13,93f67870e1be18f0328798b37faf07df,2,-1,1
14,93f67870e1be18f0328798b37faf07df,3,0,1
2,cda0e838de11a82a1802e38b8b0d9ab9,3,2,15
5,cda0e838de11a82a1802e38b8b0d9ab9,0,2,4
6,cda0e838de11a82a1802e38b8b0d9ab9,3,3,3
8,cda0e838de11a82a1802e38b8b0d9ab9,2,2,3
11,cda0e838de11a82a1802e38b8b0d9ab9,2,1,2


In [17]:
# Sample and inspect data from `existing_rag_output`
import os
import pandas as pd
import psycopg2

DB_ENV = "DATABASE_URL_PUBLIC_RAG_DR"
db_url = os.getenv(DB_ENV)
if not db_url:
    raise RuntimeError(f"Environment variable {DB_ENV} is not set")

conn = psycopg2.connect(db_url)
try:
    disinct_settings = pd.read_sql_query("""SELECT distinct config_hash, created_at, embedding_model,
                                               top_n_retrieval, semantic_weight, answer_gen_llm
                                               FROM existing_rag_output 
                                            order by semantic_weight""", conn)
finally:
    try:
        conn.close()
    except Exception:
        pass

print('existing_rag_output shape:', disinct_settings.shape)
print('Columns:', list(disinct_settings.columns))
display(disinct_settings)

existing_rag_output shape: (3, 6)
Columns: ['config_hash', 'created_at', 'embedding_model', 'top_n_retrieval', 'semantic_weight', 'answer_gen_llm']


,config_hash,created_at,embedding_model,top_n_retrieval,semantic_weight,answer_gen_llm
0,cda0e838de11a82a1802e38b8b0d9ab9,2026-04-17 02:39:05.138105,text-embedding-3-small,1,0.1,llama-3.1-8b-instant
1,fac62bbce5555de5ca44b01442baea3f,2026-04-16 00:46:56.983543,text-embedding-3-small,1,0.5,llama-3.1-8b-instant
2,93f67870e1be18f0328798b37faf07df,2026-04-16 00:48:20.342787,text-embedding-3-small,1,0.8,openai/gpt-oss-120b


## Compare 2 RAG Settings --> Provide Score Reasons

* For each record:
  * Compare 2 RAG settings
    * Statstically better, reasons why RAG2 are better than RAG1 (lessons learned)
    * Not Statistically better, reasons why RAG 1 didn't reach to score 3, why RAG 2 didn't reach to score 3 --> Summarize combined reasons

In [30]:
import importlib, utils_root_cause
importlib.reload(utils_root_cause)
from langchain_groq import ChatGroq
from utils_root_cause import run_compare_2rags_v1, run_summarize_patterns, run_why_lower_score_v1

exp_llm = ChatGroq(
    groq_api_key=os.environ["GROQ_TOKEN"],
    model_name="openai/gpt-oss-20b",
    temperature=0.78
)

In [22]:
# --- Build per-config settings string ---
def build_settings_str(config_hash):
    r = disinct_settings[disinct_settings['config_hash'] == config_hash].iloc[0]
    return ", ".join(f"{col}: {r[col]}" for col in ['embedding_model', 'top_n_retrieval', 'semantic_weight', 'answer_gen_llm'])


def build_comparison_datasets(selected_hashes, df_records_rows, disinct_settings):
    """Build same-score and diff-score comparison DataFrames for two config hashes.

    Parameters
    ----------
    selected_hashes : list[str]
        Exactly two config hash strings to compare.
    df_records_rows : pd.DataFrame
        Exploded records DataFrame containing per-hash rows.
    disinct_settings : pd.DataFrame
        DataFrame of distinct RAG config settings keyed by config_hash.

    Returns
    -------
    dataset_same_score : pd.DataFrame
    dataset_diff_score : pd.DataFrame
    """
    assert len(selected_hashes) == 2, "selected_hashes must contain exactly 2 config hashes"
    hash1, hash2 = selected_hashes

    QUERY_COL     = 'rca_records.query'
    RQ_SCORE_COL  = 'rca_records.new_retrieval_quality_score'
    AQ_SCORE_COL  = 'rca_records.new_answer_quality_score'
    RQ_REASON_COL = 'rca_records.rq_reasoning'
    AQ_REASON_COL = 'rca_records.aq_reasoning'

    rag1_settings_str = build_settings_str(hash1)
    rag2_settings_str = build_settings_str(hash2)

    # --- Subset and merge on query (no full-frame copy) ---
    keep_cols = [QUERY_COL, RQ_SCORE_COL, AQ_SCORE_COL, RQ_REASON_COL, AQ_REASON_COL]
    df_merged = (df_records_rows[df_records_rows['config_hash'] == hash1][keep_cols]
                 .merge(df_records_rows[df_records_rows['config_hash'] == hash2][keep_cols],
                        on=QUERY_COL, suffixes=('_rag1', '_rag2')))

    # --- Convert score cols to numeric in one call ---
    score_num_cols = [RQ_SCORE_COL + s for s in ('_rag1', '_rag2')] + [AQ_SCORE_COL + s for s in ('_rag1', '_rag2')]
    df_merged[score_num_cols] = df_merged[score_num_cols].apply(pd.to_numeric, errors='coerce')

    # --- Add formatted columns once on merged frame, then slice ---
    df_merged['rag1_settings'] = rag1_settings_str
    df_merged['rag2_settings'] = rag2_settings_str
    for rag, sfx in [('rag1', '_rag1'), ('rag2', '_rag2')]:
        for name, sc, rc in [('rq', RQ_SCORE_COL, RQ_REASON_COL), ('aq', AQ_SCORE_COL, AQ_REASON_COL)]:
            df_merged[f'{rag}_{name}_score_and_reasons'] = (
                '{Score: ' + df_merged[sc+sfx].astype(str) + ', Reason: ' + df_merged[rc+sfx].astype(str) + '}'
            )

    rq_diff = (df_merged[RQ_SCORE_COL + '_rag1'] - df_merged[RQ_SCORE_COL + '_rag2']).abs()
    aq_diff = (df_merged[AQ_SCORE_COL + '_rag1'] - df_merged[AQ_SCORE_COL + '_rag2']).abs()

    final_cols = ['rag1_settings', 'rag1_rq_score_and_reasons', 'rag1_aq_score_and_reasons',
                  'rag2_settings', 'rag2_rq_score_and_reasons', 'rag2_aq_score_and_reasons']
    dataset_same_score = df_merged[(rq_diff == 0) & (aq_diff == 0)][final_cols].reset_index(drop=True)
    dataset_diff_score = df_merged[(rq_diff != 0) | (aq_diff != 0)][final_cols].reset_index(drop=True)

    return dataset_same_score, dataset_diff_score


<b>Statistically Better Case</b>

* Summarize lessons learn

In [9]:
selected_hashes = ['fac62bbce5555de5ca44b01442baea3f', '93f67870e1be18f0328798b37faf07df']


dataset_same_score, dataset_diff_score = build_comparison_datasets(
    selected_hashes, df_records_rows, disinct_settings
)
print(f"dataset_same_score: {len(dataset_same_score)} records")
print(f"dataset_diff_score: {len(dataset_diff_score)} records")
display(dataset_same_score.head(3))
display(dataset_diff_score.head(3))

dataset_same_score: 9 records
dataset_diff_score: 21 records


,rag1_settings,rag1_rq_score_and_reasons,rag1_aq_score_and_reasons,rag2_settings,rag2_rq_score_and_reasons,rag2_aq_score_and_reasons
0,"embedding_model: text-embedding-3-small, top_n...","{Score: 3, Reason: The retrieved content is es...","{Score: 2, Reason: The AI’s answer correctly a...","embedding_model: text-embedding-3-small, top_n...","{Score: 3, Reason: The retrieved content is id...","{Score: 2, Reason: The AI's answer correctly e..."
1,"embedding_model: text-embedding-3-small, top_n...","{Score: 3, Reason: The retrieved content exact...","{Score: 2, Reason: The AI’s answer addresses t...","embedding_model: text-embedding-3-small, top_n...","{Score: 3, Reason: The retrieved content repea...","{Score: 2, Reason: The AI’s answer addresses t..."
2,"embedding_model: text-embedding-3-small, top_n...","{Score: 3, Reason: The retrieved content is id...","{Score: 2, Reason: The AI’s answer correctly i...","embedding_model: text-embedding-3-small, top_n...","{Score: 3, Reason: The retrieved content is es...","{Score: 2, Reason: The AI’s answer is relevant..."


,rag1_settings,rag1_rq_score_and_reasons,rag1_aq_score_and_reasons,rag2_settings,rag2_rq_score_and_reasons,rag2_aq_score_and_reasons
0,"embedding_model: text-embedding-3-small, top_n...","{Score: 3, Reason: The retrieved content inclu...","{Score: 2, Reason: The AI’s answer correctly c...","embedding_model: text-embedding-3-small, top_n...","{Score: 3, Reason: The retrieved content direc...","{Score: 3, Reason: The AI’s answer covers ever..."
1,"embedding_model: text-embedding-3-small, top_n...","{Score: 3, Reason: The retrieved content is id...","{Score: -1, Reason: The referenced answer disc...","embedding_model: text-embedding-3-small, top_n...","{Score: 3, Reason: The retrieved content is es...","{Score: 3, Reason: The AI’s answer fully addre..."
2,"embedding_model: text-embedding-3-small, top_n...","{Score: 3, Reason: The retrieved content is a ...","{Score: 2, Reason: The AI’s answer addresses t...","embedding_model: text-embedding-3-small, top_n...","{Score: 3, Reason: The retrieved content is id...","{Score: 3, Reason: The AI’s answer is fully re..."


In [10]:
df_compare_output = await run_compare_2rags_v1(dataset_diff_score, exp_llm)

print(f"Processed {len(df_compare_output)} records")
display(df_compare_output.head(3))

Processed 21 records


,rag1_settings,rag1_rq_score_and_reasons,rag1_aq_score_and_reasons,rag2_settings,rag2_rq_score_and_reasons,rag2_aq_score_and_reasons,lessons_learned
0,"embedding_model: text-embedding-3-small, top_n...","{Score: 3, Reason: The retrieved content inclu...","{Score: 2, Reason: The AI’s answer correctly c...","embedding_model: text-embedding-3-small, top_n...","{Score: 3, Reason: The retrieved content direc...","{Score: 3, Reason: The AI’s answer covers ever...",RAG2's higher semantic_weight (0.8 vs 0.5) yie...
1,"embedding_model: text-embedding-3-small, top_n...","{Score: 3, Reason: The retrieved content is id...","{Score: -1, Reason: The referenced answer disc...","embedding_model: text-embedding-3-small, top_n...","{Score: 3, Reason: The retrieved content is es...","{Score: 3, Reason: The AI’s answer fully addre...",RAG2’s higher semantic_weight (0.8 vs 0.5) kee...
2,"embedding_model: text-embedding-3-small, top_n...","{Score: 3, Reason: The retrieved content is a ...","{Score: 2, Reason: The AI’s answer addresses t...","embedding_model: text-embedding-3-small, top_n...","{Score: 3, Reason: The retrieved content is id...","{Score: 3, Reason: The AI’s answer is fully re...",RAG2 outperforms RAG1 mainly because its highe...


In [ ]:
folder_id = os.getenv('GOOGLE_DRIVE_FOLDER_ID')
upload_to_google_drive(df_compare_output, folder_id, 'compare_2rags_v1.xlsx')

In [11]:
summarized_patterns = await run_summarize_patterns(df_compare_output, exp_llm, 
                                                   text_col='lessons_learned',
                                                   pattern_focus="why RAG2's answer quality is statistically better than RAG1")

for pattern in summarized_patterns:
    print(pattern)

Higher semantic_weight (0.8 vs 0.5) tightens retrieval relevance, ensuring key facts are captured.
The larger GPT‑OSS‑120b LLM produces richer, more detailed answers, capturing nuances missed by the smaller llama‑3.1‑8b model.
Together, stronger weighting and a bigger LLM raise answer scores, while retrieval scores stay similar due to identical top‑n and embeddings.


<b>Not Statistically Better Case</b>

* Why didn't reach to score 3

In [21]:
df_records_rows['config_hash'].value_counts()

config_hash
fac62bbce5555de5ca44b01442baea3f    30
93f67870e1be18f0328798b37faf07df    30
cda0e838de11a82a1802e38b8b0d9ab9    30
Name: count, dtype: int64

In [33]:
rag1_hash = 'fac62bbce5555de5ca44b01442baea3f'
rag2_hash = 'cda0e838de11a82a1802e38b8b0d9ab9'

def build_rag_df(df_records_rows, config_hash):
    rag = df_records_rows[
        (df_records_rows['config_hash'] == config_hash) &
        (df_records_rows['rca_records.new_answer_quality_score'].isin([0, 1, 2]))
    ][['config_hash', 'rca_records.query',
       'rca_records.new_retrieval_quality_score', 'rca_records.rq_reasoning',
       'rca_records.new_answer_quality_score', 'rca_records.aq_reasoning']].copy()
    rag['rag_rq_score_and_reasons'] = rag.apply(
        lambda r: {'Score': r['rca_records.new_retrieval_quality_score'], 'Reason': r['rca_records.rq_reasoning']}, axis=1
    )
    rag['rag_aq_score_and_reasons'] = rag.apply(
        lambda r: {'Score': r['rca_records.new_answer_quality_score'], 'Reason': r['rca_records.aq_reasoning']}, axis=1
    )
    rag.drop(columns=['rca_records.new_retrieval_quality_score', 'rca_records.rq_reasoning',
                      'rca_records.new_answer_quality_score', 'rca_records.aq_reasoning'], inplace=True)
    rag['rag_settings'] = build_settings_str(config_hash)
    return rag

rag1 = build_rag_df(df_records_rows, rag1_hash)
print(rag1.shape)

rag2 = build_rag_df(df_records_rows, rag2_hash)
print(rag2.shape)

rag1.head()


(25, 5)
(26, 5)


,config_hash,rca_records.query,rag_rq_score_and_reasons,rag_aq_score_and_reasons,rag_settings
0,fac62bbce5555de5ca44b01442baea3f,How to deposit a cheque issued to an associate...,"{'Score': 3, 'Reason': 'The retrieved content ...","{'Score': 2, 'Reason': 'The AI’s answer correc...","embedding_model: text-embedding-3-small, top_n..."
1,fac62bbce5555de5ca44b01442baea3f,Can I send a money order from USPS as a business?,"{'Score': 3, 'Reason': 'The retrieved content ...","{'Score': 2, 'Reason': 'The AI’s answer correc...","embedding_model: text-embedding-3-small, top_n..."
3,fac62bbce5555de5ca44b01442baea3f,Applying for and receiving business credit,"{'Score': 3, 'Reason': 'The retrieved content ...","{'Score': 2, 'Reason': 'The AI’s answer addres...","embedding_model: text-embedding-3-small, top_n..."
4,fac62bbce5555de5ca44b01442baea3f,401k Transfer After Business Closure,"{'Score': 3, 'Reason': 'The retrieved content ...","{'Score': 2, 'Reason': 'The AI’s answer is rel...","embedding_model: text-embedding-3-small, top_n..."
5,fac62bbce5555de5ca44b01442baea3f,What are the ins/outs of writing equipment pur...,"{'Score': 3, 'Reason': 'The retrieved content ...","{'Score': 2, 'Reason': 'The AI’s answer correc...","embedding_model: text-embedding-3-small, top_n..."


In [34]:
import asyncio

rag1_insights, rag2_insights = await asyncio.gather(
    run_why_lower_score_v1(rag1, exp_llm),
    run_why_lower_score_v1(rag2, exp_llm),
)
print(f"rag1_insights: {rag1_insights.shape}, rag2_insights: {rag2_insights.shape}")
display(rag1_insights.head(3))

rag1_insights: (25, 6), rag2_insights: (26, 6)


,config_hash,rca_records.query,rag_rq_score_and_reasons,rag_aq_score_and_reasons,rag_settings,insights
0,fac62bbce5555de5ca44b01442baea3f,How to deposit a cheque issued to an associate...,"{'Score': 3, 'Reason': 'The retrieved content ...","{'Score': 2, 'Reason': 'The AI’s answer correc...","embedding_model: text-embedding-3-small, top_n...","The retrieval was perfect (score 3), so missin..."
1,fac62bbce5555de5ca44b01442baea3f,Can I send a money order from USPS as a business?,"{'Score': 3, 'Reason': 'The retrieved content ...","{'Score': 2, 'Reason': 'The AI’s answer correc...","embedding_model: text-embedding-3-small, top_n...",The answer omitted several critical details fr...
2,fac62bbce5555de5ca44b01442baea3f,Applying for and receiving business credit,"{'Score': 3, 'Reason': 'The retrieved content ...","{'Score': 2, 'Reason': 'The AI’s answer addres...","embedding_model: text-embedding-3-small, top_n...",Answer quality fell short because the model om...


In [35]:
folder_id = os.getenv('GOOGLE_DRIVE_FOLDER_ID')
upload_to_google_drive(rag1_insights, folder_id, 'lower_score_rag1_v1.xlsx')
upload_to_google_drive(rag2_insights, folder_id, 'lower_score_rag2_v1.xlsx')

Uploaded file ID: 1fu2zF33UcZzxHzGZT2Q--Dwiyt88DeCW
Uploaded file ID: 1JObEzHHkNc3Mh6C383JhPz6Gm6rcZNUx


In [36]:
df_insights = pd.concat([rag1_insights.assign(rag='rag1'), rag2_insights.assign(rag='rag2')], ignore_index=True)

summarized_patterns = await run_summarize_patterns(df_insights, exp_llm, 
                                                   text_col='insights',
                                                   pattern_focus="why RAG's answer quality is lower than 3")

for pattern in summarized_patterns:
    print(pattern)

Top‑n=1 and low semantic_weight cut context, so the LLM misses many details, producing incomplete answers.
Prompt lacks explicit instruction to reference retrieved content, causing hallucination or omission of key points.
Small, concise LLMs (e.g., llama‑3.1‑8b‑instant) truncate nuance, failing to elaborate on all critical details.
